# Second judge — cross-family Φ over the Batch API

Scores the seven main conditions' validation outputs with an OpenAI rater, so the
register metric Φ no longer rests on a single judge.

---
## 1 — Setup

API-only: no GPU, no torch.

In [2]:
!git pull

Already up to date.


In [3]:
!pip install -q openai==2.41.1 PyYAML==6.0.3 python-dotenv==1.2.2 numpy scipy


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [4]:
import getpass, logging, os
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
logging.getLogger('httpx').setLevel(logging.WARNING)

In [5]:
%cd /home/prnamhr/projects/Style-Aware-MT

/home/prnamhr/projects/Style-Aware-MT


---
## 2 — Pre-flight

Each assertion guards a way this run could corrupt the primary Φ or produce an
invalid comparison. Do not relax one to make the cell pass.

In [6]:
import json, pathlib, yaml
from src.eval.judge import judge_results_path, judge_segment_dir, template_digest
from src.eval.judge_batch import read_state, state_path
from src.eval.judge_ci import MAIN_CONDITIONS

CONFIG  = 'configs/judge_eval_gpt.yaml'
SPLIT   = 'val'
CONDS   = MAIN_CONDITIONS            # the seven main conditions
RESULTS = pathlib.Path('results')

cfg = yaml.safe_load(pathlib.Path(CONFIG).read_text())
judge, TAG = cfg['judge'], cfg['tag']

# -- a genuinely different family, or there is no cross-check
assert judge['provider'] == 'openai', judge['provider']
assert TAG, 'tag must be set, or this run overwrites the primary judge artefacts'
assert judge['model'] and judge['model'] != 'terra', \
    'set the confirmed API model id in the config -- a wrong id fails the batch'

# -- reasoning off is the whole cost argument; assert it rather than trust it
assert judge.get('reasoning_effort') == 'none', judge.get('reasoning_effort')

# -- identical frozen rubric
assert cfg['template_file'] == 'prompts/judge_eval.txt', cfg['template_file']
DIGEST = template_digest(pathlib.Path(cfg['template_file']).read_text())

# -- artefacts must not collide with the primary judge's
out_b, dir_b = judge_results_path(RESULTS, SPLIT, TAG), judge_segment_dir(RESULTS, SPLIT, TAG)
out_a = judge_results_path(RESULTS, SPLIT, None)
assert out_b != out_a
assert out_a.exists(), 'the primary judge results are missing; nothing to compare against'
a = json.loads(out_a.read_text())
models_a = sorted({v['model'] for v in a.values() if isinstance(v, dict)})
assert judge['model'] not in models_a, 'judge B is the same model as judge A'

# -- all seven conditions present, equal length, identical segments in identical order
rows = {c: [json.loads(x) for x in pathlib.Path(f'outputs/{c}_{SPLIT}.jsonl')
            .read_text().splitlines() if x.strip()] for c in CONDS}
n_eval = len(rows[CONDS[0]])
for c in CONDS:
    assert len(rows[c]) == n_eval, f'{c}: {len(rows[c])} rows, expected {n_eval}'
    assert [r['input'] for r in rows[c]] == [r['input'] for r in rows[CONDS[0]]], \
        f'{c} segments diverge from {CONDS[0]}'

print(f'conditions   : {len(CONDS)}  {CONDS}')
print(f'segments each: {n_eval}   (test split sealed)')
print(f'judge A      : {models_a}')
print(f'judge B      : {judge["model"]}  tag={TAG}  reasoning={judge["reasoning_effort"]}')
print(f'rubric       : {cfg["template_file"]} [{DIGEST}]  (same file as judge A)')
print(f'writes to    : {out_b}  and  {dir_b}/')

inflight = read_state(state_path(RESULTS, SPLIT, TAG))
print(f'in-flight batches: {inflight or "none"}')
if inflight:
    print('  -> a previous submission is still running. Re-running section 5 RESUMES')
    print('     polling it. It does NOT resubmit, so you are not billed twice.')

/home/prnamhr/projects/Style-Aware-MT/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


conditions   : 7  ['zeroshot', 'random_fewshot', 'knn_fewshot', 'afsp_margin', 'afsp_full', 'peft', 'commercial_haiku']
segments each: 1323   (test split sealed)
judge A      : ['claude-haiku-4-5']
judge B      : gpt-5.6-terra  tag=gpt  reasoning=none
rubric       : prompts/judge_eval.txt [ffd6dad41acb0512]  (same file as judge A)
writes to    : results/judge_gpt_val.json  and  results/judge_gpt_val_segments/
in-flight batches: none


---
## 3 — Cost

With reasoning off, ~89% of the token volume is input, so this is an input bill and
the estimate is unusually tight. It is still an estimate: the tokenizer treats
Persian and Arabic source text worse than English.

In [ ]:
tpl = pathlib.Path(cfg['template_file']).read_text()
chars = sum(len(tpl) + len(r['input']) + len(r['output']) + len(r.get('prediction', ''))
            for c in CONDS for r in rows[c])
calls = len(CONDS) * n_eval
D = judge.get('batch_discount', 0.5)
rates = judge.get('pricing') or [None, None]

print(f'calls {calls:,}')
for div in (3.0, 3.5, 4.0):
    print(f'  @{div} chars/tok -> {chars/div/1e6:5.2f}M input tokens')
print(f'  output ~{60*calls/1e6:.2f}M tokens at ~60 tok/call (reasoning off)')

IN, OUT = chars/3.5/1e6, 60*calls/1e6
if None in rates:
    print('\npricing is not set in the config, so no dollar figure can be given.')
    print('Cost = %.2f x in_rate + %.2f x out_rate, times the %.2f batch discount:' % (IN, OUT, D))
    for ir in (0.15, 0.40, 1.00, 1.25, 2.00):
        print(f'  list ${ir:>4.2f}/${ir*4:>5.2f} per 1M  ->  ${D*(IN*ir + OUT*ir*4):6.2f}')
else:
    est = D*(IN*rates[0] + OUT*rates[1])
    print(f'\nESTIMATE at list ${rates[0]}/${rates[1]} per 1M, batch x{D}: ${est:.2f}')
    print(f'  (without batch: ${IN*rates[0] + OUT*rates[1]:.2f})')
print('\nAnchor: the Claude judge measured $0.6316 for 1,323 calls'
      f' -> ${0.6316/1323*calls:.2f} for this volume.')

---
## 4 — Pilot

Settles the two things the estimate above cannot: real tokens per call, and whether
the model honours `reasoning_effort: none`. Costs about a cent.

`--limit` writes the segment cache but deliberately **not** a results file, so a
pilot can never be mistaken for a scored condition. The full run resumes from these
25 segments at no extra charge.

In [ ]:
!python3 manage.py judge_batch --conditions zeroshot --split {SPLIT} \
    --config {CONFIG} --limit 25 --poll_interval 15

In [ ]:
u = json.loads((RESULTS / f'judge_{TAG}_{SPLIT}_usage.json').read_text())
s = u['session']
if s['calls'] == 0:
    print('no calls billed this session (cache already complete)')
else:
    per = {k: s[k]/s['calls'] for k in ('prompt_tokens','completion_tokens','cost_usd')}
    print(f"pilot: {s['calls']} calls, {s['prompt_tokens']:,} in / {s['completion_tokens']:,} out")
    print(f"       {per['prompt_tokens']:.0f} in / {per['completion_tokens']:.0f} out per call")
    if per['completion_tokens'] > 150:
        print('  !! far above the ~60 expected -- the model is likely still emitting')
        print('     reasoning tokens. Re-check reasoning_effort before the full run.')
    if not u['priced']:
        print('  !! no pricing configured; cost_usd is a floor of 0, not real spend')
    else:
        print(f"\nPROJECTED FULL PASS: {calls:,} calls  ~${per['cost_usd']*calls:.2f}")
print('\nConfirm against the budget cap before section 5.')

---
## 5 — Full pass

One batch per condition, 24-hour completion window. Safe to interrupt: the batch id
is persisted before polling starts, so re-running this cell **resumes** the
in-flight job rather than submitting a second one.

If a batch expires, only the contiguous scored prefix is written — the unwritten
tail is resubmitted on the next run rather than frozen into the cache as nulls.

In [ ]:
CONDS_ARG = ' '.join(CONDS)
!python3 manage.py judge_batch --conditions {CONDS_ARG} --split {SPLIT} --config {CONFIG}

In [ ]:
b = json.loads(out_b.read_text())
print(f"{'condition':<18}{'n':>6}{'coverage':>10}{'Phi_B':>8}{'Phi_A':>8}{'B-A':>8}")
for c in CONDS:
    rb, ra = b.get(c), a.get(c)
    if not rb:
        print(f'{c:<18}  MISSING'); continue
    d = rb['mean'] - ra['mean'] if ra and ra.get('mean') and rb.get('mean') else float('nan')
    print(f"{c:<18}{rb['n']:>6}{rb['coverage']:>10.4f}{rb['mean']:>8.3f}"
          f"{(ra['mean'] if ra else float('nan')):>8.3f}{d:>+8.3f}")

errs = {c: sum('error' in json.loads(x)
               for x in (dir_b / f'{c}.jsonl').read_text().splitlines() if x.strip())
        for c in CONDS if (dir_b / f'{c}.jsonl').exists()}
print('\nfailed calls per condition:', {k: v for k, v in errs.items() if v} or 'none')
print('to retry failures: delete those lines from the segment cache and re-run section 5')

---
## 6 — Standalone Φ table

This rater reported as a metric in its own right: per-condition Φ with percentile
intervals, the distribution over the 1–5 rubric, a ranking, and the bootstrap rank
distribution behind that ranking.

Read the **rank distribution**, not just the ranking. `P(this rank)` near 1/k means
the position is not established, however clean the ordering looks.

In [ ]:
!python3 manage.py judge_ci --tag {TAG} --split {SPLIT} --n_resamples 10000

In [ ]:
# The same table for the primary judge, for context. Separate artefact, not merged.
!python3 manage.py judge_ci --split {SPLIT} --n_resamples 10000

---
## 7 — Judge–judge agreement

Secondary to §6. Contrasts are held to one common segment set across both raters,
so a difference between the two columns is rater identity and nothing else.

In [ ]:
!python3 manage.py judge_agreement --tag_b {TAG} --split {SPLIT} --n_resamples 10000

In [ ]:
rep = json.loads((RESULTS / f'judge_agreement_{TAG}_{SPLIT}.json').read_text())
pooled = rep['rater_agreement']['study_only']['pooled']
print(f"pooled n={pooled['n']}  qwk={pooled['qwk']['kappa']:+.3f}"
      f"  rho={pooled['spearman']['rho']:+.3f}"
      f"  exact={pooled['exact_agreement']:.1%}  adjacent={pooled['adjacent_agreement']:.1%}")
print(f"severity offset A-B = {pooled['offset']['diff']:+.3f}"
      f" [{pooled['offset']['ci_low']:+.3f}, {pooled['offset']['ci_high']:+.3f}]")
print(f"identical system ranking: {rep['condition_ordering']['study_only']['identical_ranking']}")

print('\nContrasts by stability:')
for v_name in ('RATER-DEPENDENT', 'both separate', 'neither separates'):
    names = [k for k, v in rep['contrast_replication']['contrasts'].items()
             if (('both separate' if v['both_separate'] else
                  'neither separates' if v['neither_separates'] else
                  'RATER-DEPENDENT') == v_name)]
    print(f'  {v_name:<18} {names or "-"}')
flipped = [k for k, v in rep['contrast_replication']['contrasts'].items() if not v['same_sign']]
print(f'\nsign flips between raters: {flipped or "none"}')

---
## 8 — What may be reported

**From §6, the standalone table**

- Φ under this rater is its own metric column. Do **not** average it with the
  primary Φ: if the two raters differ in severity, a combined column is
  uninterpretable.
- Adjacent-rank gaps are **uncorrected**. Apply a family-wise correction before
  calling any of them significant.
- An unstarred neighbouring pair is not distinguishable; its relative order is not
  a finding.
- `commercial_haiku` is a diagnostic external reference, not a condition of the
  study, under either rater. Keep it out of study tables.

**From §7, the agreement pass**

- `both separate` (Holm-corrected under each rater) is robust to rater identity.
  Report with both intervals.
- `RATER-DEPENDENT` separates under one rater only — not reportable without naming
  the rater it depends on.
- `neither separates` is a failure to separate, **not** evidence of no difference.
  The detection floor still applies.
- A **sign flip** means the direction itself is not established.
- A high κ does not validate Φ. Two similarly-biased raters would also agree.

**Reproducibility**

- Φ from this rater is **not** byte-reproducible either. `seed` is best-effort, so
  the second rater removes single-family dependence but not non-determinism.

**On completion, update:** `docs/DEVLOG.md` (what ran, actual spend from
`results/judge_<tag>_<split>_usage.json`, what it showed), the threats table in
`README.md`, and `docs/budget.md` with the cumulative figure.